# OIM3600 — PEMS Homework: Investigate, Validate, and Transform Market Data

**Goal:** Write your own Python code to investigate three external data sources, test their quality and consistency, and build the `market_data` dictionary used by PEMS.

**What you are given:** The external JSON files and the basic data-loading code. **What you must write:** All eight validation checks and the code to construct `market_data`. No completed validation function is provided.

**Submission:** Save this notebook with your completed Python code cells and their executed output. **All student answers must be in code cells.** Use `print()` statements to report findings and Python comments to explain reasoning. Do not add written answers in Markdown cells or paste a prewritten validation function; develop each check yourself.

## Preparation — Load the external data

Place the notebook where the data directory can be found. The cell below searches for a `data` folder in the notebook's current working directory or its parent. Run it before starting the investigation. The existing `load_data()` function in PEMS contains additional PM and menu handling; this assignment focuses on the three market-data sources.

In [ ]:
import json
from pathlib import Path
from pprint import pprint as pp
from decimal import Decimal

DATA = Path.cwd().parent.parent / "10-26M15/Classwork/data"
print(DATA)

with open(DATA / "nyse_top100.json", encoding="utf-8") as file:
    tickers = json.load(file)["tickers"]
with open(DATA / "corp_acts_yf.json", encoding="utf-8") as file:
    corporate_actions = json.load(file)["data"]
with open(DATA / "raw_eod_yf.json", encoding="utf-8") as file:
    eod_prices = json.load(file)["data"]

print("External sources loaded.")

## Part A — External data investigation and validation

Write code in **each section**. Output enough evidence to support your conclusions. When a test fails, identify the records or values responsible whenever the question asks for exceptions. Checks should **report** issues rather than terminate the notebook. Do not assume that the data is correct just because the JSON files load successfully.

### Check 1 — Verify the stock universe

Write code to:
- Report the number of tickers and the number of unique tickers.
- Determine whether there are **exactly 100 unique tickers**.
- Print each duplicated ticker, if any.

**In your code:** Add a Python comment explaining why checking only `len(tickers) == 100` is insufficient.

In [ ]:
# Check 1

# ============================================================
# INVESTIGATING AND VALIDATING OUR EXTERNAL DATA
# ============================================================
#
# Before constructing market_data, we need to understand
# the structure, completeness, consistency, and quality
# of our source data.
#
# Our sources include:
#
# tickers:
#     A list containing our universe of 100 stock symbols.
#
# corporate_actions:
#     A dictionary keyed by ticker.
#     Each ticker has a list of corporate-action records.
#
# eod_prices:
#     A dictionary keyed by ticker.
#     Each ticker has a list of daily price records.
#
# We will establish several important DATA INVARIANTS
# that our market_data construction can rely upon.
#
# A program that runs without errors does not necessarily
# mean its data is correct.
#
# ============================================================


# ============================================================
# 1. UNIVERSE - VERIFY OUR 100 UNIQUE TICKERS
# ============================================================
#
# tickers is a list containing our stock symbols.
#
# Example:
# tickers = ["TSM", "AAPL", "MSFT", ...]
#
# A set is a collection that cannot contain duplicates.
#
# Converting the list to a set eliminates any duplicate
# ticker symbols.
#
# Since we already know tickers contains 100 elements,
# a resulting set of length 100 proves there are no
# duplicate tickers.

unique_tickers = set(tickers)

# id() returns the identity of a Python object.
#
# The two IDs should be different because set(tickers)
# creates a new object rather than modifying the list.
#
# len(unique_tickers) tells us how many distinct
# ticker symbols we have.

print(f"1. Universe - tickers list ID: {id(tickers)},\n"
      f"unique set ID: {id(unique_tickers)},\n"
      f"tickers: {len(tickers)}\n"
      f"unique tickers: {len(unique_tickers)}\n")

# len(tickers) == 100 alone is insufficient: a list of 100 could contain
# duplicates (e.g., 99 distinct tickers with one listed twice).

### Check 2 — Corporate-action ticker coverage

Write code to determine whether the keys of `corporate_actions` contain exactly the tickers in the stock universe. Print a Boolean result, any **missing** tickers, and any **extra** tickers.

**Explain:** What does a missing ticker mean for the transformation?

In [ ]:
# Check 2

# ============================================================
# 2. CORPORATE-ACTION COVERAGE
# ============================================================
#
# corporate_actions is a dictionary.
#
# Example:
#
# corporate_actions = {
#     "TSM": [{...}, {...}],
#     "AAPL": [{...}],
#     "MSFT": [],
#     ...
# }
#
# The dictionary keys identify the tickers for which
# corporate-action information is available.
#
# set(corporate_actions) creates a set of its keys.
#
# This is equivalent to:
# set(corporate_actions.keys())
#
# IMPORTANT:
# A ticker can exist in corporate_actions even if its
# associated list is empty. An empty list means there
# are no corporate-action records for that ticker.

ca_tickers = set(corporate_actions)

# Set equality checks that both sets contain exactly
# the same elements, regardless of their order.
#
# True means:
#   - Every ticker in our universe is represented.
#   - No unexpected tickers are present.
#
# We are establishing a ticker-coverage invariant.

print(f"2. Corporate-action ticker coverage matches universe: {ca_tickers == unique_tickers}\n")

# A missing ticker matters for the transformation: when building
# market_data we look up corporate_actions[ticker] for every ticker.
# A missing key would raise a KeyError, or, if we skipped it, we
# would silently assume no dividends or splits for that stock.
# We'd have no way to tell "no actions" apart from "no data."

### Check 3 — EOD ticker coverage

Compare the keys of `eod_prices` with the stock universe. Print a Boolean result and any missing or extra tickers. Do not reuse the answer from Check 2; investigate this source separately.

In [ ]:
# Check 3

# ============================================================
# 3. END-OF-DAY (EOD) COVERAGE
# ============================================================
#
# eod_prices is also a dictionary keyed by ticker.
#
# Example:
#
# eod_prices = {
#     "TSM": [{...}, {...}, ...],
#     "AAPL": [{...}, {...}, ...],
#     ...
# }
#
# Each value is a list of daily price records.
#
# Once again, converting the dictionary to a set
# produces a set containing its ticker keys.

eod_tickers = set(eod_prices)

# Verify that the EOD dictionary contains exactly
# the same ticker symbols as our universe.
#
# Together, checks 2 and 3 establish that:
#
# unique_tickers == ca_tickers == eod_tickers
#
# Therefore, all three sources cover the same
# 100 ticker symbols.
#
# We do not need to repeat these coverage checks
# during market_data construction.

print(f"3. EOD ticker coverage matches universe: {eod_tickers == unique_tickers}\n")

### Check 4 — EOD observation counts

For each ticker in the EOD dictionary, count its records. Report:
- How many tickers have **exactly 100** observations.
- How many do not.
- The ticker and actual count for every exception.

**Explain:** Why does the transformation depend on consistent observation counts?

In [ ]:
# Check 4

# ============================================================
# 4. OBSERVATION COUNT - 100 DAYS PER TICKER
# ============================================================
#
# Our EOD data should contain exactly 100 observations
# for every ticker.
#
# eod_prices.values() returns the values associated
# with the ticker keys. Each ticker key's value is a list.
#
# Each value is a list of daily price dictionaries.
#
# Example:
#
# records = [
#     {"Date": "...", "Close": 238.88, ...},
#     {"Date": "...", "Close": 240.15, ...},
#     ...
# ]
#
# We count how many tickers have exactly 100 records
# and how many do not.

have_100_obs = 0
not_100_obs = 0

for records in eod_prices.values():

    # records is the list of EOD observations
    # for ONE ticker.
    #
    # len(records) counts the number of observations.

    if len(records) == 100:
        have_100_obs += 1
    else:
        not_100_obs += 1

# Expected result: 100 0
#
# This establishes that every ticker has the
# expected number of observations.
#
# This matters because market_data reads every
# ticker's list by the same position (day 0..99).
# A list with fewer than 100 records would raise
# an IndexError or shift prices onto the wrong dates.
#
# However, 100 observations does NOT establish
# that they represent the same 100 dates. If the
# dates also match (Check 5), we have a price for
# every ticker on every day.
#
# That requires a separate investigation.

print(f"4. Observation counts - tickers with 100: {have_100_obs}, tickers without 100: {not_100_obs}\n")

### Check 5 — Date consistency, ordering, and uniqueness

Write code for **all four** tasks:

**5A.** Choose the first ticker in `tickers` and create a reference list of dates from its EOD records. Use the first 10 characters of each record's `"Date"` field.

**5B.** Compare the complete date sequence for every ticker with that reference sequence. Count matches and nonmatches. For each nonmatch, report the ticker and either its first differing index/date or a length mismatch.

**5C.** Determine whether the reference dates are in ascending order.

**5D.** Determine whether the reference dates are unique. Print duplicated dates if any.

**Explain:** Why is matching observation counts not enough to guarantee alignment?

In [ ]:
# Check 5A — reference date sequence

# ============================================================
# 5. DATE CONSISTENCY, ORDERING, AND UNIQUENESS
# ============================================================
#
# Having 100 observations per ticker is not enough.
#
# We also need to know:
#
# A. Do all tickers have the same dates?
# B. Are those dates in the same order?
# C. Are the dates in ascending chronological order?
# D. Are there any duplicate dates?
#
# These checks matter because our market_data
# construction will use the SAME list index (day)
# to retrieve records from every ticker.
#
# Two tickers can each have 100 records and still
# disagree: one might be missing a date and have an
# extra date somewhere else. Matching counts cannot
# detect that.
#
# We must establish that index 0, index 1, etc.
# represent the same dates across all tickers.


# ------------------------------------------------------------
# 5A. BUILD THE REFERENCE DATE SEQUENCE
# ------------------------------------------------------------
#
# We use the first ticker in our universe, tickers[0],
# as the reference.
#
# Each EOD "Date" is a timestamp string such as:
#   "2025-08-15 00:00:00-04:00"
#
# record['Date'][:10] keeps only the first 10 characters:
#   "2025-08-15"
#
# This YYYY-MM-DD format is what market_data will use
# for its "date" field, so we standardize on it here.

reference_dates = []

for record in eod_prices[tickers[0]]:
    reference_dates.append(record['Date'][:10])

print(f"5A. Reference ticker: {tickers[0]}, dates: {len(reference_dates)}, "
      f"first: {reference_dates[0]}, last: {reference_dates[-1]}\n")

In [ ]:
# Check 5B — compare the sequences across tickers

# ------------------------------------------------------------
# 5B. COMPARE EVERY TICKER WITH THE REFERENCE
# ------------------------------------------------------------
#
# For each ticker, build its date list the same way
# and compare it with reference_dates.
#
# Comparing two lists with == checks both:
#   - Their contents
#   - Their order
#
# Therefore, two lists containing the same dates
# in different orders will NOT compare equal.
#
# We loop over tickers (not eod_prices.values()) so
# that we can name any ticker that fails.

same = 0
not_same = 0

for ticker in tickers:

    # Build this ticker's list of dates.
    dates = []
    for record in eod_prices[ticker]:
        dates.append(record['Date'][:10])

    if dates == reference_dates:
        same += 1
    else:
        not_same += 1

        # Report how this ticker differs: a different
        # length, or the first position where it differs.
        if len(dates) != len(reference_dates):
            print(f"   {ticker}: length {len(dates)} "
                  f"vs reference {len(reference_dates)}")
        else:
            for i in range(len(dates)):
                if dates[i] != reference_dates[i]:
                    print(f"   {ticker}: first difference at index {i}: "
                          f"{dates[i]} vs {reference_dates[i]}")
                    break

# Expected result: 100 0
#
# If all 100 lists match, every ticker has the same
# dates in the same order. We now need to examine only
# reference_dates to establish ordering and uniqueness
# for all tickers.

print(f"5B. Date alignment - matching: {same}, nonmatching: {not_same}\n")

In [ ]:
# Check 5C — chronological order

# ------------------------------------------------------------
# 5C. CHECK ASCENDING DATE ORDER
# ------------------------------------------------------------
#
# sorted() creates a new list in ascending order.
#
# Comparing the original list with the sorted list
# tells us whether it was already in order.
#
# YYYY-MM-DD strings sort alphabetically in the same
# order as they sort chronologically, because the
# largest unit (year) comes first.
#
# Expected result: True

print(f"5C. Dates in ascending order: {reference_dates == sorted(reference_dates)}\n")

In [ ]:
# Check 5D — duplicate dates

# ------------------------------------------------------------
# 5D. CHECK FOR DUPLICATE DATES
# ------------------------------------------------------------
#
# A set cannot contain duplicate elements.
#
# As we walk through the dates, seen holds every date
# we have already encountered. A date that is already
# in seen is a duplicate.

seen = set()
duplicates = []

for date in reference_dates:
    if date in seen:
        duplicates.append(date)
    else:
        seen.add(date)

# Expected result: True, []

print(f"5D. No duplicate dates: {len(duplicates) == 0}, duplicates: {duplicates}\n")

### Check 6 — Closing-price quality

Inspect the `"Close"` field of **every EOD record** and separately count:
- Missing `"Close"` keys
- Values equal to `None`
- Nonnumeric values (consider Python `bool` values nonnumeric for this purpose)
- Zero prices
- Negative prices

Print the ticker, date, and offending value for each problem. Print the five final counts.

**Explain:** Why might a numerically valid value still be unsuitable for a stock price?

In [ ]:
# Check 6

# ============================================================
# 6. PRICE QUALITY
# ============================================================
#
# Check the "Close" value of every EOD record for:
#
#   missing      - the "Close" key does not exist
#   none         - the value is None
#   not_numeric  - the value is not an int or float
#                  (type(True) is bool, so True/False
#                  count as nonnumeric)
#   nan          - the value is float('nan'), "not a number"
#   zero         - the price equals 0
#   negative     - the price is less than 0
#
# We check None and nonnumeric values BEFORE the
# numeric comparisons; comparing None or a string
# with 0 would cause a TypeError.

missing = 0
none = 0
not_numeric = 0
nan = 0
zero = 0
negative = 0

# Loop over tickers so each problem can be reported
# with its ticker and date.

for ticker in tickers:
    for record in eod_prices[ticker]:

        date = record['Date'][:10]
        price = record.get('Close')

        # Only one condition can be true for each record.

        if 'Close' not in record:
            missing += 1
            print(f"   {ticker} {date}: Close missing")
        elif price is None:
            none += 1
            print(f"   {ticker} {date}: Close is None")
        elif type(price) not in (int, float):
            not_numeric += 1
            print(f"   {ticker} {date}: nonnumeric Close {price}")

        # NaN is the only value that is not equal to itself,
        # so price != price is True only when price is NaN.
        # NaN is a float, is not 0, and is not negative, so
        # without this test it would pass every other check.

        elif price != price:
            nan += 1
            print(f"   {ticker} {date}: Close is NaN")
        elif price == 0:
            zero += 1
            print(f"   {ticker} {date}: zero Close")
        elif price < 0:
            negative += 1
            print(f"   {ticker} {date}: negative Close {price}")

# Expected result: 0 0 0 0 0 0
#
# A numerically valid value can still be unsuitable.
# A positive number can also be wrong, such as 2.39
# instead of 239.0.

print(f"6. Price quality - missing: {missing}, None: {none}, "
      f"nonnumeric: {not_numeric}, NaN: {nan}, zero: {zero}, "
      f"negative: {negative}\n")

### Check 7 — Corporate-action exploration

Inspect each ticker's corporate-action records. The supplied source uses `"Dividends"`, `"Stock Splits"`, and `"Date"` fields.

Write code to report:
- The distribution of dividend-event counts by ticker, using a six-element list for **0, 1, 2, 3, 4, and 5 or more** events.
- The corresponding distribution of split-event counts.
- The number of tickers that have **both** at least one dividend event and at least one split event.
- The number of distinct corporate-action dates.
- Every date with more than one corporate-action record, together with its record count.
- Whether each six-element distribution sums to 100.

Treat a nonzero `"Dividends"` as a dividend event and a nonzero `"Stock Splits"` as a split event.

**Explain:** What do the six positions in your distribution lists represent? Why is the last position different from the first five?

In [ ]:
# Check 7

# ============================================================
# 7. CORPORATE-ACTION EXPLORATION
# ============================================================
#
# corporate_actions is a dictionary keyed by ticker.
# Each value is a list of corporate-action records.
# Some tickers have no actions; others have several.
#
# We want to understand:
#
# A. How many dividends does each ticker have?
# B. How many stock splits does each ticker have?
# C. How many tickers have BOTH dividends and splits?
# D. On which dates do corporate-action records appear,
#    and how many records fall on each date?


# ------------------------------------------------------------
# 7A. PREPARE OUR COUNTERS
# ------------------------------------------------------------
#
# num_divs is a frequency distribution:
#
#   Index 0 counts tickers with zero dividends.
#   Index 1 counts tickers with one dividend.
#   ...
#   Index 5 counts tickers with five OR MORE dividends.
#
# The first five positions each mean an exact count.
# The last position is different: it groups every
# count of five or more, so the list stays six long.
#
# num_splits uses the same structure for stock splits.

num_divs = [0, 0, 0, 0, 0, 0]
num_splits = [0, 0, 0, 0, 0, 0]

# Number of tickers with at least one dividend
# AND at least one stock split.

both = 0

# date_counts maps each corporate-action date to the
# number of records on that date.
#
# We use a dictionary, not a set, because a set only
# records THAT a date appeared, not HOW MANY times.

date_counts = {}


# ------------------------------------------------------------
# 7B. EXAMINE EVERY TICKER'S CORPORATE ACTIONS
# ------------------------------------------------------------

for ticker, records in corporate_actions.items():

    # records is the list of corporate actions
    # for ONE ticker. Reset the counts for each ticker.

    div_cnt = 0
    splt_cnt = 0

    for record in records:

        # A nonzero value counts as an event.
        # .get(..., 0) treats a missing key as no event.

        if record.get('Dividends', 0) != 0:
            div_cnt += 1

        if record.get('Stock Splits', 0) != 0:
            splt_cnt += 1

        # Count this record under its date.
        # [:10] keeps YYYY-MM-DD to match the EOD
        # reference dates from Check 5.
        #
        # .get(date, 0) returns 0 the first time a
        # date is seen, so we can always add 1.

        date = record['Date'][:10]
        date_counts[date] = date_counts.get(date, 0) + 1

    # We have now finished this ticker's records.
    # If it had at least one dividend AND at least one
    # split, count it in both. The two events do not
    # need to occur on the same day.

    if div_cnt > 0 and splt_cnt > 0:
        both += 1

    # Group counts of five or more into index 5 so the
    # count is always a valid index (0 through 5).

    if div_cnt > 5:
        div_cnt = 5

    if splt_cnt > 5:
        splt_cnt = 5

    # Each ticker adds exactly one to each distribution.

    num_divs[div_cnt] += 1
    num_splits[splt_cnt] += 1

# ------------------------------------------------------------
# 7C. TOTAL EVENTS
# ------------------------------------------------------------
#
# Each distribution counts TICKERS, so it sums to 100.
# To count EVENTS, weight each position by the number
# of events it represents: index x number of tickers.
#
# Position 5 means "5 or more", so these totals are
# exact only when num_divs[5] and num_splits[5] are 0.

divs = 0
for idx, num in enumerate(num_divs):
    divs += idx * num

splits = 0
for idx, num in enumerate(num_splits):
    splits += idx * num


# ------------------------------------------------------------
# 7D. REPORT
# ------------------------------------------------------------

print("7. Corporate actions")
print(f"   dividend frequency: {num_divs}, sums to 100: {sum(num_divs) == 100}")
print(f"   split frequency:    {num_splits}, sums to 100: {sum(num_splits) == 100}")
print(f"   total dividends:    {divs}")
print(f"   total splits:       {splits}")
print(f"   tickers with both:  {both}")
print(f"   distinct corporate-action dates: {len(date_counts)}")

# Dates shared by more than one record, in date order.

print("   dates with more than one record:")
for date in sorted(date_counts):
    if date_counts[date] > 1:
        print(f"      {date}: {date_counts[date]} records")
print()

### Check 8 — Corporate-action date alignment

Compare the set of corporate-action dates with the set of EOD dates from Check 5. Report whether **every corporate-action date** occurs in the EOD sequence; print any missing dates.

**Explain:** Why would a corporate action on a date not represented in the EOD source complicate construction of `market_data`?

In [ ]:
# Check 8

# ============================================================
# 8. CORPORATE-ACTION DATE ALIGNMENT
# ============================================================
#
# We have already established that:
#
# 1. The EOD and corporate-action dictionaries contain
#    exactly the same ticker symbols. (Checks 2, 3)
#
# 2. Every ticker has 100 EOD observations. (Check 4)
#
# 3. Every ticker has the same EOD dates as
#    reference_dates, in the same order. (Check 5B)
#
# 4. The reference dates are ascending and unique.
#    (Checks 5C, 5D)
#
# Therefore, reference_dates represents the EOD dates
# for all 100 tickers.
#
# We now check whether every corporate-action date
# also appears in the EOD date sequence.
#
# ca_dates:  the corporate-action dates. The keys of
#            date_counts (Check 7) are exactly these.
# eod_dates: the reference dates as a set.
#
# Both use YYYY-MM-DD, so they can be compared directly.

ca_dates = set(date_counts)
eod_dates = set(reference_dates)

# Set difference: the dates in ca_dates that are NOT
# in eod_dates. An empty result means every
# corporate-action date appears in the EOD sequence.

missing_dates = ca_dates - eod_dates

# Expected result: True, []
#
# WHY A MISSING DATE WOULD COMPLICATE market_data:
#
# market_data is built one trading day at a time from
# the EOD dates. A corporate action on a date that is
# not an EOD date has no trading day to attach to.
# It would be silently dropped (a dividend never paid,
# a split never applied), unless we invented a rule
# for moving it to another day.

print(f"8. Corporate-action dates all present in EOD: {len(missing_dates) == 0}, "
      f"missing: {sorted(missing_dates)}\n")

## Part B — Construct `market_data`

**Objective:** Reorganize end-of-day (EOD) data by trading day and combine it with the corresponding corporate actions. Before writing the transformation, study the new dictionary's shape and how to access its values.

### B1. The fundamental change: ticker-first to day-first

**Before:** `eod_prices` is organized by **ticker**, then observation/day.  
**After:** `market_data` is organized by **trading day**, then ticker.

```text
BEFORE: eod_prices                  AFTER: market_data
Organized by TICKER                 Organized by TRADING DAY

eod_prices                          market_data
|                                   |
+-- TSM                              +-- Day 0
|   +-- Day 0: Price                |   +-- date: 2025-08-15
|   +-- Day 1: Price                |   +-- TSM: Price, Div, Split
|   +-- Day 2: Price                |   +-- AAPL: Price, Div, Split
|   +-- ...                         |   +-- ... 98 more tickers
|   +-- Day 99: Price               |
|                                   +-- Day 1
+-- AAPL                             |   +-- date: 2025-08-18
|   +-- Day 0: Price                |   +-- TSM: Price, Div, Split
|   +-- Day 1: Price                |   +-- AAPL: Price, Div, Split
|   +-- Day 2: Price                |   +-- ... 98 more tickers
|   +-- ...                         |
|   +-- Day 99: Price               +-- ...
|                                   |
+-- ... 98 more tickers             +-- Day 99
```

The original access pattern is:

```python
eod_prices[ticker][day]["Close"]
```

Here `day` refers to a **position in a ticker's list of daily observations**, assuming the records are aligned and ordered (as investigated in Part A).

The new access pattern is:

```python
market_data[day][ticker]["price"]
market_data[day][ticker]["div"]
market_data[day][ticker]["split_in"]
market_data[day][ticker]["split_out"]
```

**What changes?** We start with 100 tickers, each containing 100 daily observations, and reorganize them into 100 trading days, each containing 100 ticker records **plus a date field**. We also merge in dividend and stock-split information from `corporate_actions`; this is more than simply reversing two indexes.

**Why?** PEMS advances one trading day at a time. With the day-first structure, the system can retrieve all information for a given trading day from one daily dictionary.

### B2. The target nested-dictionary structure

This is a **small illustrative excerpt**. The finished `market_data` must have all 100 tickers on each of days `0` through `99`.

```python
market_data = {
    0: {
        "date": "2025-08-15",
        "TSM": {"price": 238.88, "div": 0, "split_in": 1, "split_out": 1},
        "BRK.B": {"price": 477.20, "div": 0, "split_in": 1, "split_out": 1},
        "LLY": {"price": 701.23, "div": 1.5, "split_in": 1, "split_out": 1},
        # ... 97 additional tickers
    },
    1: {
        "date": "2025-08-18",
        "TSM": {"price": 241.41, "div": 0, "split_in": 1, "split_out": 1},
        "BRK.B": {"price": 478.52, "div": 0, "split_in": 1, "split_out": 1},
        "LLY": {"price": 698.05, "div": 0, "split_in": 1, "split_out": 1},
        # ... 97 additional tickers
    },
    # ... through day 99
}
```

The example is provided to **explain the intended shape**, not as data to copy into the solution. Retrieve dates, prices, dividends, and split details from the files.

**Three levels of dictionary access:**

```text
market_data
  +-- 0  (trading day)
  |   +-- "date" --> "2025-08-15"
  |   +-- "TSM"  --> {"price": ..., "div": ..., "split_in": ..., "split_out": ...}
  |   +-- "LLY"  --> {"price": ..., "div": ..., "split_in": ..., "split_out": ...}
  |   +-- ... 98 more tickers
  +-- 1  (next trading day)
  +-- ... through day 99
```

- **Level 1:** `market_data[day]` returns the **daily-record dictionary**.
- **Level 2:** `market_data[day][ticker]` returns the **ticker-record dictionary** for that day.
- **Level 3:** `market_data[day][ticker]["price"]` returns a **single numeric value**. Substitute `"div"`, `"split_in"`, or `"split_out"` to retrieve those fields.

Each daily-record dictionary also has a `"date"` key: `market_data[day]["date"]`. Thus, it contains **101 keys**, not 100: one date and 100 tickers. Unlike the ticker values, the date value is a string.

### B3. Reading nested dictionaries one step at a time

For the example above, let `day = 0` and `ticker = "LLY"`.

| Expression | What it returns | Type |
|---|---|---|
| `market_data` | Dictionary of all trading days | `dict` |
| `market_data[day]` | Date plus all 100 ticker records for day 0 | `dict` |
| `market_data[day]["date"]` | `"2025-08-15"` | `str` |
| `market_data[day][ticker]` | `{"price": 701.23, "div": 1.5, "split_in": 1, "split_out": 1}` | `dict` |
| `market_data[day][ticker]["price"]` | `701.23` | `float` |
| `market_data[day][ticker]["div"]` | `1.5` | `float` |

Python accesses these dictionaries **one key at a time**. The following expressions are equivalent:

```python
price = market_data[day][ticker]["price"]
```

```python
daily_record = market_data[day]
ticker_record = daily_record[ticker]
price = ticker_record["price"]
```

**Important:** Do not treat every key inside `market_data[day]` as a ticker. The key `"date"` maps to a string, not a ticker-record dictionary. Code that tries `market_data[day]["date"]["price"]` will fail.

### B4. Plan the transformation before coding

Decide how to solve these questions. Explain your choices **in Python comments in the implementation cell**, not in new Markdown answer cells.

1. Should the outer loop move through tickers or trading days? Why?
2. When will you create each new daily-record dictionary, and where will you get its date?
3. How will you retrieve the right EOD closing price for a ticker and trading day?
4. How will you match corporate actions to the **same ticker and date**?
5. What values will you use when there is no dividend or split?
6. How will you avoid treating `"date"` as a ticker?

Your code should build the dictionary from the source data; it should not hard-code the illustrative records.

### B5. Implementation requirements

Using the loaded `tickers`, `corporate_actions`, and `eod_prices`, write Python code to create a dictionary named `market_data`.

**Required structure** (illustrative values only):

```python
{
    0: {
        "date": "2025-08-15",
        "TSM": {"price": 238.88, "div": 0, "split_in": 1, "split_out": 1},
        "BRK.B": {"price": 477.20, "div": 0, "split_in": 1, "split_out": 1},
        # ... all 100 tickers
    },
    1: {
        "date": "2025-08-18",
        # ... all 100 tickers
    },
    # ... through trading day 99
}
```

Requirements:
1. Use **integer** trading-day keys from `0` through `99`, in chronological order.
2. Each day must contain a `"date"` string in `YYYY-MM-DD` format and an entry for **every ticker**.
3. Each ticker's `"price"` is that day's EOD `"Close"` value.
4. Attach the dividend and split information for that **ticker and date**. Use `div = 0`, `split_in = 1`, `split_out = 1` when no applicable action exists.
5. Examine the corporate-action records before implementing split conversion. Use Python comments in your transformation cell to explain how you interpret `"Stock Splits"` and produce `split_in` and `split_out`, including any limitations of the source representation. Do not invent action values.
6. Build the dictionary from the files; do **not** type in the example prices or dates.

Write the transformation in the following cell. You may create small helper functions if useful.

In [ ]:
# Part B — implement the transformation


# ==============================================================
# ==============================================================
# REORGANIZING EOD DATA INTO market_data
# ==============================================================
#
# BEFORE: eod_prices                 AFTER: market_data
# Organized by TICKER                Organized by TRADING DAY
#
# eod_prices                         market_data
# |                                  |
# +-- TSM                             +-- Day 0
# |   +-- Day 0: Price               |   +-- date: 2025-08-15
# |   +-- Day 1: Price               |   +-- TSM: Price, Div, Split
# |   +-- Day 2: Price               |   +-- AAPL: Price, Div, Split
# |   +-- ...                        |   +-- ... 98 more tickers
# |   +-- Day 99: Price              |
# |                                  +-- Day 1
# +-- AAPL                            |   +-- date: 2025-08-18
# |   +-- Day 0: Price               |   +-- TSM: Price, Div, Split
# |   +-- Day 1: Price               |   +-- AAPL: Price, Div, Split
# |   +-- Day 2: Price               |   +-- ... 98 more tickers
# |   +-- ...                        |
# |   +-- Day 99: Price              +-- ...
# |                                  |
# +-- ... 98 more tickers            +-- Day 99
#
#
# ==============================================================
# THE FUNDAMENTAL CHANGE
# ==============================================================
#
# BEFORE:
#     TICKER -> DAY -> PRICE RECORD
#
#     eod_prices[ticker][day]['Close']
#
# AFTER:
#     DAY -> TICKER -> CONSOLIDATED RECORD
#
#     market_data[day][ticker]['price']
#                             ['div']
#                             ['split_in']
#                             ['split_out']
#
# We are taking:
#     100 tickers, each containing 100 daily records
#
# And reorganizing them into:
#     100 trading days, each containing 100 ticker records.
#
# In addition to reorganizing the EOD prices, we incorporate
# dividend and stock-split information from corporate_actions.
#
# WHY?
# Our portfolio system processes information one trading day
# at a time. This new structure gives us all the information
# needed for a particular trading day in one dictionary.
#
# ==============================================================




# ============================================================
# CONSTRUCT market_data
# ============================================================
#
# GOAL:
# Combine end-of-day (EOD) prices and corporate actions into
# one dictionary organized by trading day.
#
# market_data has the following structure:
#
# market_data = {
#     0: {
#         "date": "2025-08-15",
#         "TSM": {
#             "price": 238.88,
#             "div": 0.0,
#             "split_in": 1,
#             "split_out": 1
#         },
#         "AAPL": {...},
#         ... remaining tickers
#     },
#     1: {
#         "date": "2025-08-18",
#         "TSM": {...},
#         "AAPL": {...},
#         ...
#     },
#     ... through day 99
# }
#
# There are three levels:
#
# Level 1: market_data[day] -> daily_record
# Level 2: daily_record[ticker] -> ticker_record
# Level 3: ticker_record -> price, div, split_in, split_out
#
# Each daily_record also contains a "date" field.
#
# ============================================================

market_data = {}

# ------------------------------------------------------------
# OUTER LOOP: PROCESS EACH TRADING DAY
# ------------------------------------------------------------
#
# range(100) generates integers 0 through 99.
#
# Each iteration creates one daily_record containing:
#   - The calendar date
#   - A ticker_record for each of our 100 tickers
#
# We use tickers[0] to obtain the date because our earlier
# investigation established that all 100 tickers have exactly
# the same dates in exactly the same order.
#
# This is an established DATA INVARIANT.
#
# For testing, consider using range(5).

for day in range(100):

    # Create a new dictionary for the current trading day.
    #
    # Example when day = 0:
    #
    # eod_prices[tickers[0]] -> list of 100 records
    # [day]                  -> first record in that list
    # ['Date']               -> full date/time string
    # [:10]                  -> calendar date only
    #
    # daily_record initially contains only the date.
    # The inner loop will add all 100 ticker records.

    daily_record = {
        'date': eod_prices[tickers[0]][day]['Date'][:10]
    }

    # --------------------------------------------------------
    # INNER LOOP: PROCESS EVERY TICKER FOR THIS DAY
    # --------------------------------------------------------
    #
    # For every trading day, visit all 100 tickers.
    #
    # Because our earlier validation established that the
    # EOD records are aligned, index [day] refers to the
    # same calendar date for every ticker.
    #
    # For testing, consider using tickers[:5].

    for ticker in tickers:

        # ----------------------------------------------------
        # CREATE THE TICKER RECORD
        # ----------------------------------------------------
        #
        # eod_prices[ticker] is a list of 100 dictionaries.
        #
        # eod_prices[ticker][day] selects the dictionary
        # for this ticker on the current trading day.
        #
        # ['Close'] retrieves its closing stock price.
        #
        # Initially assume that NO corporate action occurred.
        #
        # div = 0       -> no cash dividend
        # split_in = 1  -> one share before the split
        # split_out = 1 -> one share after the split
        #
        # A 1-for-1 split means no change in share ownership.
        #
        # If a corporate action occurred, we will update
        # these default values below.

        ticker_record = {
            'price': eod_prices[ticker][day]['Close'],
            'div': 0,
            'split_in': 1,
            'split_out': 1
        }

        # ----------------------------------------------------
        # SEARCH FOR CORPORATE ACTIONS ON THIS DATE
        # ----------------------------------------------------
        #
        # corporate_actions is a dictionary keyed by ticker.
        #
        # corporate_actions[ticker] is a LIST of dictionaries.
        #
        # Example:
        #
        # corporate_actions = {
        #     "TSM": [
        #         {
        #             "Date": "2025-09-18 00:00:00-04:00",
        #             "Dividends": 0.73,
        #             "Stock Splits": 0
        #         },
        #         {...}
        #     ],
        #     "AAPL": [...],
        #     ...
        # }
        #
        # Each iteration assigns ONE corporate-action record
        # for the current ticker to the variable action.
        #
        # Our earlier data investigation established that
        # corporate-action records are ordered by date.
        #
        # We will use this ordering to stop searching early.

        for action in corporate_actions[ticker]:

            # Extract just the calendar date.
            # Example:
            # "2025-09-18 00:00:00-04:00"
            # becomes "2025-09-18".

            action_date = action['Date'][:10]

            # ------------------------------------------------
            # CASE 1: CORPORATE ACTION DATE MATCHES TODAY
            # ------------------------------------------------
            #
            # We found a corporate-action record for the
            # ticker on the date we are currently processing.
            #
            # Update the ticker_record with the event data.

            if action_date == daily_record['date']:

                # Read the dividend per share.
                #
                # .get('Dividends', 0) returns:
                #   - The dividend value if the key exists
                #   - 0 if the key is missing
                #
                # Therefore, a date with no dividend retains
                # the default value of zero.

                ticker_record['div'] = action.get(
                    'Dividends', 0
                )

                # Read the stock split factor.
                #
                # A value of 0 means no split occurred.
                # A nonzero value represents the split ratio.
                #
                # Examples:
                #   5.0  -> 5 new shares for 1 old share
                #   0.5  -> 1 new share for 2 old shares

                split_factor = action.get('Stock Splits', 0)

                # If a split occurred, convert the split
                # factor into a numerator and denominator.
                #
                # Example:
                #   Decimal('5.0').as_integer_ratio()
                #   returns (5, 1)
                #
                #   Decimal('0.5').as_integer_ratio()
                #   returns (1, 2)
                #
                # split_out = numerator (new shares)
                # split_in  = denominator (old shares)
                #
                # str() is used before Decimal() to preserve
                # the decimal representation of the factor
                # rather than exposing binary float artifacts.

                if split_factor:
                    ticker_record['split_out'], ticker_record['split_in'] = (
                        Decimal(str(split_factor)).as_integer_ratio()
                    )

                # We found the record for this ticker/date.
                #
                # Under our assumption of at most one
                # corporate-action record per ticker/date,
                # there is no reason to continue searching.
                #
                # break exits ONLY the corporate-action loop.
                # It does not exit the ticker or day loops.

                break

            # ------------------------------------------------
            # CASE 2: CORPORATE ACTION DATE IS IN THE FUTURE
            # ------------------------------------------------
            #
            # Suppose we are processing August 20 and the
            # next corporate action is September 18.
            #
            # Since the corporate-action list is ordered
            # chronologically, all subsequent records must
            # also be after August 20.
            #
            # Therefore, no matching record can exist later
            # in the list.
            #
            # We can stop searching immediately.
            #
            # If no event was found, our ticker_record
            # retains its default corporate-action values.

            if action_date > daily_record['date']:
                break

        # ----------------------------------------------------
        # ADD THE COMPLETED TICKER RECORD TO DAILY_RECORD
        # ----------------------------------------------------
        #
        # Example:
        #
        # daily_record["TSM"] = {
        #     "price": 238.88,
        #     "div": 0,
        #     "split_in": 1,
        #     "split_out": 1
        # }
        #
        # After processing all 100 tickers, daily_record
        # contains 101 keys:
        #   - 1 date key
        #   - 100 ticker keys

        daily_record[ticker] = ticker_record

    # --------------------------------------------------------
    # ADD THE COMPLETED DAILY RECORD TO MARKET_DATA
    # --------------------------------------------------------
    #
    # At this point, the ticker loop has finished.
    #
    # daily_record now contains the date and all 100
    # ticker records for the current trading day.
    #
    # Store it using day (an integer from 0 to 99) as
    # the dictionary key.
    #
    # Example:
    # market_data[0] = daily_record for August 15, 2025
    #
    # The next iteration creates a NEW daily_record for
    # the next trading day.

    market_data[day] = daily_record

for day in range(3):
    print(f"Day {day}:")
    pp(market_data[day], sort_dicts=False)

        
        
    

### Test your constructed dictionary

Write your **own** tests, not just a visual inspection. At minimum, show that:
- The day keys are `0` through `99`.
- The dates correspond to the EOD reference sequence.
- Each day contains exactly 100 ticker entries plus the `"date"` field.
- Each ticker record has exactly the four required fields.
- Selected prices match the source EOD records.
- Corporate-action values match the source on dates with actions, and defaults appear on dates without actions.

Print a small sample of your result (for example, five tickers on days 0 and 1) rather than all 10,000 ticker-day records.

In [ ]:
# Write your market_data tests here
